##Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info_raw")

# Silver Transformations

## Finding Duplicates

In [0]:
duplicates = (
    df.groupBy(df.columns)
        .count()
        .filter(F.col("count") > 1)
)
print(True if duplicates.count() > 0 else False)

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Product Key Parsing

In [0]:
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

##Product Line Normalization

In [0]:

df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)
display(df)

##Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity Check of Dataframe

In [0]:
df.limit(10).display()

#Writting to Silver Table

In [0]:
(
    df.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable('workspace.silver.crm_products')
)


##Sanity Check of Silver Table

In [0]:
%sql
SELECT *
FROM workspace.silver.crm_products
LIMIT 10